# JustBuild Foundation - Lead Qualification Agent

Twenty-five inbound enquiries, one qualification policy, one scoring rubric. This notebook runs the same **three-agent** design you drew on the board, now pointed at the sales pipeline:

**Input -> Agent 1 (Ingest & Extract) -> Agent 2 (Judge & Prioritise) -> Agent 3 (Communicate) -> You (the human)**

- **Agent 1** reads each enquiry and pulls out the fit signals.
- **Agent 2** scores the lead against the rubric, bands it, and routes it. The weights, the bands, and the disqualify **hard rule** all live in **code**, not in the model.
- **Agent 3** drafts a personalised follow-up and assigns the next action. It **contacts nobody** - that stays your call.

Everything the notebook needs - the enquiries, the qualification policy, and the **OC Brain** - is pulled from the hosted pack. Nothing is pasted inline.

> Run the cells top to bottom. Watch for the green checkmarks.

In [ ]:
#@title 1 · Setup  { display-mode: "form" }
!pip -q install google-generativeai pypdf requests pandas >/dev/null 2>&1
import requests, io, re, json, time
from pypdf import PdfReader
import pandas as pd
print('Setup complete - libraries ready.')

In [ ]:
#@title 2 · Configuration  { display-mode: "form" }
import getpass

MODE = "Live Gemini"  #@param ["Live Gemini", "Switch to OC Brain"]
MODEL = "gemini-2.5-flash-lite"  #@param {type:"string"}

BASE = "https://justbuild.orangecaterpillar.com/foundation-material/lead-qualification"
LEADS_URL    = f"{BASE}/Leads.pdf"
POLICY_URL   = f"{BASE}/Qualification_Policy.pdf"
OC_BRAIN_URL = f"{BASE}/oc-brain.json"

GEMINI_API_KEY = ""
if MODE == "Live Gemini":
    GEMINI_API_KEY = getpass.getpass('Paste your Gemini API key (hidden). Leave blank to use the OC Brain: ').strip()

print(f'Mode: {MODE}   Model: {MODEL}')

In [ ]:
#@title 3 · Load the leads, the policy, and the OC Brain  { display-mode: "form" }
def fetch_pdf_pages(url):
    r = requests.get(url, timeout=30); r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return [(p.extract_text() or '') for p in reader.pages]

lead_pages = fetch_pdf_pages(LEADS_URL)
LEAD_BLOCKS = []
for t in lead_pages:
    if re.search(r'Lead \d+ of \d+', t):
        LEAD_BLOCKS.append(re.sub(r'Lead \d+ of \d+\s*', '', t).strip())

POLICY_TEXT = '\n'.join(fetch_pdf_pages(POLICY_URL))
OC_BRAIN = requests.get(OC_BRAIN_URL, timeout=30).json()

# Read the band routing (owner + next action) out of the policy. Fall back to the OC Brain's copy.
ROUTING = {}
for band, owner, action in re.findall(r'ROUTE:\s*(Hot|Warm|Nurture|Disqualify)\s*\|\s*(.+?)\s*\|\s*(.+)', POLICY_TEXT):
    ROUTING[band] = (owner.strip(), action.strip())
if len(ROUTING) < 3:
    ROUTING = {r['band']: (r['owner'], r['next_action']) for r in OC_BRAIN['routing_table']}

print(f'Loaded {len(LEAD_BLOCKS)} leads and {len(ROUTING)} band routes from the hosted pack.')
print(f"OC Brain on standby - {len(OC_BRAIN['leads'])} prepared results loaded.")

In [ ]:
#@title 4 · The engine (and the OC Brain resilience layer)  { display-mode: "form" }
USE_OC_BRAIN = (MODE == 'Switch to OC Brain') or (not GEMINI_API_KEY)
if USE_OC_BRAIN and MODE == 'Live Gemini':
    print('No API key given - running on the OC Brain.')

model = None
if not USE_OC_BRAIN:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(MODEL)

def call_gemini(prompt, tries=4):
    last = None
    for i in range(tries):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            last = e; msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg: what = '429 (busy / free-tier limit)'
            elif '503' in msg or 'UNAVAILABLE' in msg:      what = '503 (model overloaded)'
            elif '404' in msg:                              what = '404 (model name)'
            else:                                           what = 'an error'
            wait = 2 ** i
            print(f'Gemini returned {what} - try {i+1}/{tries}. Waiting {wait}s...')
            time.sleep(wait)
    raise last

def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    start = min([i for i in (text.find('['), text.find('{')) if i != -1])
    return json.loads(text[start:])

def switch_to_oc_brain(reason):
    global USE_OC_BRAIN
    print(f'{reason} Switching to the OC Brain to keep going.')
    USE_OC_BRAIN = True

print('Engine ready.', '(OC Brain)' if USE_OC_BRAIN else f'(Live: {MODEL})')

## Agent 1 - Ingest & Extract
Turn 25 enquiries into one clean table of fit signals. Live, Gemini reads each enquiry; on the OC Brain, the same facts come from the prepared file.

In [ ]:
#@title 5 · Agent 1 runs  { display-mode: "form" }
records = []  # one row per lead, in pipeline order

if not USE_OC_BRAIN:
    try:
        numbered = '\n\n'.join(f'--- LEAD {i+1} ---\n{t}' for i, t in enumerate(LEAD_BLOCKS))
        prompt = (
            'You are Agent 1 in a sales pipeline. We sell a workflow-automation platform to business teams. '
            'For EACH enquiry below, return ONLY a JSON array, in the same order (no prose), with exactly '
            'these fields:\n'
            '  lead_id (string), company (string), contact (string), role (string),\n'
            '  seniority ("exec" a decision-maker, "manager", "champion" an individual advocate, or "unknown"),\n'
            '  company_size ("enterprise", "mid", "smb", or "unknown"),\n'
            '  budget ("stated", "unclear", or "none"),\n'
            '  need_match ("strong", "some", "weak", or "none" - how well their need fits workflow automation),\n'
            '  timeline ("now", "quarter", "exploring", or "none"),\n'
            '  spam (true if it is not a genuine buying enquiry: a job application, a vendor pitch, or a student/research request),\n'
            '  summary (a short phrase).\n\n' + numbered
        )
        for d in extract_json(call_gemini(prompt)):
            records.append({'lead_id': d.get('lead_id'), 'company': d.get('company'), 'contact': d.get('contact'),
                            'role': d.get('role'),
                            'extract': {k: d.get(k) for k in ('seniority','company_size','budget','need_match','timeline','spam','summary')}})
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')

if USE_OC_BRAIN:
    records = []
    for ld in OC_BRAIN['leads']:
        ex = ld['extract']
        records.append({'lead_id': ex['lead_id'], 'company': ex['company'], 'contact': ex['contact'], 'role': ex['role'],
                        'extract': {k: ex.get(k) for k in ('seniority','company_size','budget','need_match','timeline','spam','summary')}})

df1 = pd.DataFrame([{'Lead': r['lead_id'], 'Company': r['company'], 'Seniority': r['extract']['seniority'],
                     'Size': r['extract']['company_size'], 'Budget': r['extract']['budget'],
                     'Fit': r['extract']['need_match'], 'Timeline': r['extract']['timeline'], 'Spam': r['extract']['spam']} for r in records])
print(f'Agent 1 extracted {len(df1)} leads:')
df1

## Agent 2 - Judge & Prioritise  (rules in code)
Score each lead against the rubric, band it, route it, and order the pipeline. The **weights**, the band thresholds, and the disqualify **hard rule** all live in **code** - the scoring model is the policy, not the model's guess.

In [ ]:
#@title 6 · Agent 2 runs  { display-mode: "form" }
# OUR rubric and thresholds, enforced here in code.
WEIGHTS = {'need_match': 0.35, 'authority': 0.20, 'budget': 0.20, 'timeline': 0.15, 'company_size': 0.10}
POINTS = {
    'need_match': {'strong': 10, 'some': 6, 'weak': 3, 'none': 0},
    'authority':  {'exec': 10, 'manager': 7, 'champion': 5, 'unknown': 2},
    'budget':     {'stated': 10, 'unclear': 5, 'none': 1},
    'timeline':   {'now': 10, 'quarter': 7, 'exploring': 4, 'none': 1},
    'company_size': {'enterprise': 10, 'mid': 7, 'smb': 4, 'unknown': 3},
}
RANK = {'Hot': 1, 'Warm': 2, 'Nurture': 3, 'Disqualify': 4}

def score_of(x):
    n = POINTS['need_match'].get(x.get('need_match'), 0)
    a = POINTS['authority'].get(x.get('seniority'), 2)
    b = POINTS['budget'].get(x.get('budget'), 1)
    t = POINTS['timeline'].get(x.get('timeline'), 1)
    s = POINTS['company_size'].get(x.get('company_size'), 3)
    return round((n*WEIGHTS['need_match'] + a*WEIGHTS['authority'] + b*WEIGHTS['budget']
                  + t*WEIGHTS['timeline'] + s*WEIGHTS['company_size']) * 10, 1)
def band_of(x, score):
    if x.get('spam'): return 'Disqualify'                                          # hard rule
    if score >= 75 and x.get('seniority') != 'unknown' and x.get('timeline') in ('now', 'quarter'): return 'Hot'
    if score >= 55: return 'Warm'
    if score >= 35: return 'Nurture'
    return 'Disqualify'
def evidence_of(x, score, band):
    if x.get('spam'): return 'Not a genuine sales lead.'
    if band == 'Hot': return 'Strong fit, a decision-maker, and a near-term timeline.'
    if band == 'Warm':
        if score >= 75 and x.get('timeline') not in ('now', 'quarter'): return 'Strong profile, but no near-term timeline yet.'
        if score >= 75 and x.get('seniority') == 'unknown': return 'Strong profile, but no named decision-maker yet.'
        return 'A promising lead worth a direct follow-up.'
    if band == 'Nurture': return 'Some interest, but not sales-ready yet.'
    return 'Weak fit, no clear need or timeline.'

for r in records:
    ex = r['extract']
    r['score'] = score_of(ex); r['band'] = band_of(ex, r['score'])
    r['owner'], r['action'] = ROUTING.get(r['band'], ('Frontline', 'Follow up'))
    r['evidence'] = evidence_of(ex, r['score'], r['band'])

# prioritise the pipeline: best band first, then highest score
pipeline = sorted(records, key=lambda r: (RANK[r['band']], -r['score']))

show = pd.DataFrame([{'#': i+1, 'Lead': r['lead_id'], 'Score': r['score'], 'Band': r['band'],
                      'Owner': r['owner'], 'Why': r['evidence']} for i, r in enumerate(pipeline)])
from collections import Counter
counts = Counter(r['band'] for r in pipeline)
print('Agent 2 qualified the pipeline (rubric applied in code):', {k: counts[k] for k in ['Hot','Warm','Nurture','Disqualify'] if k in counts})
show

In [ ]:
#@title    ...the Hot leads, front and centre  { display-mode: "form" }
HOT = [r for r in pipeline if r['band'] == 'Hot']
print(f'WORK FIRST - {len(HOT)} Hot lead(s):\n')
for r in HOT:
    print(f"  {r['lead_id']}  {r['company']}  ({r['score']}/100)  ->  {r['owner']}  [{r['action']}]")
    print(f"     {r['evidence']}\n")

## Agent 3 - Communicate  (behind a human gate)
Draft a personalised follow-up for each Hot lead and assign the next action for the rest. The agent **drafts**; it contacts no one. You approve and send.

In [ ]:
#@title 7 · Agent 3 drafts (nothing is sent)  { display-mode: "form" }
draft = None
if not USE_OC_BRAIN:
    try:
        hot = [{'lead_id': r['lead_id'], 'company': r['company'], 'contact': r['contact'], 'role': r['role'],
                'owner': r['owner'], 'action': r['action'], 'summary': r['extract'].get('summary')} for r in HOT]
        rest = [{'lead_id': r['lead_id'], 'company': r['company'], 'band': r['band'], 'score': r['score'],
                 'owner': r['owner'], 'action': r['action']} for r in pipeline if r['band'] != 'Hot']
        prompt = (
            'You are Agent 3 in a sales pipeline for a workflow-automation platform. Write a short handover note '
            'to the sales manager. For each Hot lead, include a brief, personalised follow-up message to the '
            'contact (warm, specific, no over-promising) and the owner it is assigned to. Then summarise the rest '
            'of the pipeline by band. Make clear that nobody has been contacted and that sending is theirs to '
            'approve. Return only the note.\n\n' + json.dumps({'hot': hot, 'rest': rest})
        )
        draft = call_gemini(prompt)
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')
if USE_OC_BRAIN or draft is None:
    draft = OC_BRAIN['draft']

print('=' * 70)
print(draft)
print('=' * 70)
print('\nHUMAN GATE: these are drafts. The agent has contacted no one. You decide what goes out.')

In [ ]:
#@title 8 · Your decision  { display-mode: "form" }
DECISION = "Hold"  #@param ["Hold", "Approve the follow-ups (I will send them myself)", "Reject"]
if DECISION.startswith('Approve'):
    print('You approved the follow-ups. The agent still contacts no one - you send them yourself, deliberately.')
elif DECISION == 'Reject':
    print('Rejected. Nothing leaves this notebook.')
else:
    print('On hold. Nobody is contacted. The decision stays with you.')

## Keep the result
Save the qualified pipeline and the drafts to files you can download or paste into your notes, and compare the agent's pipeline with the one the room worked out by hand.

In [ ]:
#@title 9 · Save the qualified pipeline  { display-mode: "form" }
lines = ['# Qualified pipeline - Sales Ops Lead Qualification', '',
         f'Mode: {"OC Brain" if USE_OC_BRAIN else MODEL}', '', '## Pipeline (best first)', '']
for i, r in enumerate(pipeline, 1):
    lines.append(f"{i}. **{r['lead_id']}** [{r['band']} {r['score']}] - {r['company']} -> {r['owner']} ({r['action']}) - {r['evidence']}")
lines += ['', '## Handover and drafted follow-ups', '', '```', draft, '```']
md_out = '\n'.join(lines)

with open('lead_pipeline.md', 'w') as f: f.write(md_out)
with open('lead_results.json', 'w') as f:
    json.dump({'mode': 'oc_brain' if USE_OC_BRAIN else MODEL,
               'pipeline': [{'lead_id': r['lead_id'], 'score': r['score'], 'band': r['band'],
                             'owner': r['owner'], 'action': r['action'], 'why': r['evidence']} for r in pipeline],
               'draft': draft}, f, indent=2)

print('Saved: lead_pipeline.md and lead_results.json (open them from the file panel on the left).')
print('\n----- copy from here -----\n')
print(md_out)
# To download instead, uncomment:
# from google.colab import files; files.download('lead_pipeline.md')

## What you just saw
The same three narrow agents, chained: extract, judge, communicate - the judge is a scoring rubric in pure **code**, so every lead is weighed the same way, and the final call still belongs to a **human**. Swap this pack for your own and the pipeline serves a different job, which is exactly what you'll do next with your own use-case.